In [5]:
import java.io.File
import java.nio.file.{Files, Paths}
import scala.util.matching.Regex

// ============================================================
// CONFIG
// ============================================================

val inputDir =
  new File("/home/vishnu/Desktop/BDA/NEON_chem-surfacewater")

val outputDir =
  new File("/home/vishnu/Desktop/BDA/NEON_Cleaned_By_Site")


// ============================================================
// CREATE OUTPUT DIRECTORY
// ============================================================

if (!inputDir.exists()) {
  println("❌ INPUT DIRECTORY DOES NOT EXIST!")
  println(inputDir.getAbsolutePath)

} else if (!inputDir.isDirectory) {
  println("❌ INPUT PATH IS NOT A DIRECTORY!")

} else {

  // Force creation
  if (!outputDir.exists()) {
    val created = outputDir.mkdirs()

    if (created)
      println(s"✅ Created output directory: ${outputDir.getAbsolutePath}")
    else
      println("❌ Could not create output directory!")
  }

  println()
  println("Input  : " + inputDir.getAbsolutePath)
  println("Output : " + outputDir.getAbsolutePath)
}


// ============================================================
// USEFUL DATA FILES
// ============================================================

val usefulFiles = Seq(
  "swc_fieldData",
  "swc_externalLabDataByAnalyte"
)


// ============================================================
// FILES TO IGNORE
// ============================================================

val ignoredFiles = Seq(
  "categoricalCodes",
  "validation",
  "variables",
  "readme",
  "EML",
  "manifest",
  "externalLabBlanks",
  "externalLabSummaryData"
)


// ============================================================
// RECURSIVELY FIND FILES
// ============================================================

def getAllFiles(dir: File): Seq[File] = {

  val files = Option(dir.listFiles()).getOrElse(Array.empty)

  files.flatMap { file =>

    if (file.isDirectory)
      getAllFiles(file)
    else
      Seq(file)

  }
}


// ============================================================
// CHECK USEFUL CSV
// ============================================================

def isUsefulCSV(file: File): Boolean = {

  val name = file.getName.toLowerCase

  name.endsWith(".csv") &&
  usefulFiles.exists(x => name.contains(x.toLowerCase)) &&
  !ignoredFiles.exists(x => name.contains(x.toLowerCase))
}


// ============================================================
// EXTRACT SITE ID FROM FILE NAME
//
// Example:
// NEON.D01.HOPB.DP1.20093.001...
//
// Result:
// D01_HOPB
// ============================================================

def extractSiteID(fileName: String): Option[String] = {

  val pattern =
    """NEON\.(D\d{2})\.([A-Za-z0-9]+)\..*""".r

  fileName match {

    case pattern(domain, site) =>
      Some(domain + "_" + site)

    case _ =>
      None
  }
}


// ============================================================
// FIND SITE COLUMN
// ============================================================

def findSiteColumn(columns: Array[String]): Option[String] = {

  val possibleNames = Seq(
    "siteID",
    "siteId",
    "site_id",
    "site"
  )

  possibleNames
    .find(name => columns.contains(name))
    .orElse {

      columns.find(
        _.toLowerCase.contains("siteid")
      )

    }
}


// ============================================================
// REMOVE UNWANTED COLUMNS
//
// NOTE:
// Keep siteID + actual measurement columns.
// Remove obvious metadata / geographic information.
// ============================================================

def isUnwantedColumn(column: String): Boolean = {

  val c = column.toLowerCase

  val unwanted = Seq(

    // Metadata
    "uid",
    "recordid",
    "dataqaqc",
    "release",
    "publication",
    "remarks",
    "comments",

    // Geographic / regional information
    "latitude",
    "longitude",
    "lat",
    "lon",
    "easting",
    "northing",
    "utm",
    "state",
    "country",
    "county",
    "region",
    "city",

    // Location descriptions
    "location",
    "namedlocation",
    "description",

    // Instrument / processing metadata
    "instrument",
    "sensor",
    "method",
    "protocol",
    "calibration"
  )

  unwanted.exists(x => c.contains(x))
}


// ============================================================
// SIMPLE CSV PARSER
// Handles normal comma-separated NEON CSVs
// ============================================================

def parseCSVLine(line: String): Array[String] = {

  val result = scala.collection.mutable.ArrayBuffer[String]()

  val current = new StringBuilder
  var insideQuotes = false

  line.foreach { ch =>

    ch match {

      case '"' =>
        insideQuotes = !insideQuotes

      case ',' if !insideQuotes =>
        result += current.toString.trim
        current.clear()

      case _ =>
        current.append(ch)
    }
  }

  result += current.toString.trim

  result.toArray
}


// ============================================================
// ESCAPE CSV VALUES
// ============================================================

def escapeCSV(value: String): String = {

  if (
    value.contains(",") ||
    value.contains("\"") ||
    value.contains("\n")
  ) {

    "\"" +
      value.replace("\"", "\"\"") +
      "\""

  } else {
    value
  }
}


// ============================================================
// PROCESS DATA
// ============================================================

if (inputDir.exists() && inputDir.isDirectory) {

  val allFiles =
    getAllFiles(inputDir)

  val csvFiles =
    allFiles.filter(isUsefulCSV)

  println()
  println("============================================")
  println("SCAN COMPLETE")
  println("============================================")

  println(
    s"Total files found : ${allFiles.length}"
  )

  println(
    s"Useful CSV files  : ${csvFiles.length}"
  )

  println()


  // ========================================================
  // PROCESS EACH CSV
  // ========================================================

  csvFiles.foreach { file =>

    println("--------------------------------------------")
    println("Processing:")
    println(file.getName)


    try {

      val source =
        scala.io.Source.fromFile(
          file,
          "UTF-8"
        )

      val lines =
        source.getLines().toArray

      source.close()


      if (lines.length > 0) {

        // --------------------------------------------------
        // HEADER
        // --------------------------------------------------

        val header =
          parseCSVLine(lines(0))


        println(
          s"Columns found: ${header.length}"
        )


        // --------------------------------------------------
        // FIND SITE COLUMN
        // --------------------------------------------------

        val siteColumn =
          findSiteColumn(header)


        // --------------------------------------------------
        // DETERMINE SITE FROM FILE NAME
        // --------------------------------------------------

        val fileSiteID =
          extractSiteID(file.getName)


        println(
          "Site from filename: " +
          fileSiteID.getOrElse("NOT FOUND")
        )


        // --------------------------------------------------
        // SELECT USEFUL COLUMNS
        // --------------------------------------------------

        val selectedColumns =
          header.zipWithIndex.filter {

            case (column, index) =>

              !isUnwantedColumn(column) ||
              siteColumn.contains(column)

          }


        val selectedIndexes =
          selectedColumns.map(_._2)


        val selectedHeader =
          selectedColumns.map(_._1)


        println(
          s"Columns retained: ${selectedHeader.length}"
        )


        // --------------------------------------------------
        // SITE ID
        // --------------------------------------------------

        val siteID =
          fileSiteID.getOrElse("UNKNOWN_SITE")


        // --------------------------------------------------
        // CREATE SITE DIRECTORY
        // --------------------------------------------------

        val siteFolder =
          new File(
            outputDir,
            siteID
          )

        if (!siteFolder.exists()) {

          val created =
            siteFolder.mkdirs()

          if (created)
            println(
              s"✅ Created: ${siteFolder.getAbsolutePath}"
            )
          else
            println(
              s"❌ Failed to create: ${siteFolder.getAbsolutePath}"
            )
        }


        // --------------------------------------------------
        // OUTPUT FILE
        // --------------------------------------------------

        val outputFile =
          new File(
            siteFolder,
            file.getName
          )


        // --------------------------------------------------
        // WRITE CLEAN CSV
        // --------------------------------------------------

        val writer =
          new java.io.PrintWriter(
            outputFile,
            "UTF-8"
          )


        // Header
        writer.println(
          selectedHeader
            .map(escapeCSV)
            .mkString(",")
        )


        // Data rows
        lines.drop(1).foreach { line =>

          val row =
            parseCSVLine(line)


          val selectedRow =
            selectedIndexes.map { index =>

              if (index < row.length)
                escapeCSV(row(index))
              else
                ""

            }


          writer.println(
            selectedRow.mkString(",")
          )
        }


        writer.close()


        println(
          s"✅ Saved: ${outputFile.getAbsolutePath}"
        )

      }

    } catch {

      case e: Exception =>

        println(
          s"❌ ERROR: ${e.getMessage}"
        )
    }
  }


  println()
  println("============================================")
  println("        PROCESSING COMPLETE")
  println("============================================")

  println(
    "Output directory:"
  )

  println(
    outputDir.getAbsolutePath
  )

}

✅ Created output directory: /home/vishnu/Desktop/BDA/NEON_Cleaned_By_Site

Input  : /home/vishnu/Desktop/BDA/NEON_chem-surfacewater
Output : /home/vishnu/Desktop/BDA/NEON_Cleaned_By_Site

SCAN COMPLETE
Total files found : 8831
Useful CSV files  : 1107

--------------------------------------------
Processing:
NEON.D19.CARI.DP1.20093.001.swc_fieldData.2016-09.expanded.20260106T144241Z.csv
Columns found: 29
Site from filename: D19_CARI
Columns retained: 26
✅ Created: /home/vishnu/Desktop/BDA/NEON_Cleaned_By_Site/D19_CARI
✅ Saved: /home/vishnu/Desktop/BDA/NEON_Cleaned_By_Site/D19_CARI/NEON.D19.CARI.DP1.20093.001.swc_fieldData.2016-09.expanded.20260106T144241Z.csv
--------------------------------------------
Processing:
NEON.D02.POSE.DP1.20093.001.swc_fieldData.2024-08.expanded.20260106T145922Z.csv
Columns found: 29
Site from filename: D02_POSE
Columns retained: 26
✅ Created: /home/vishnu/Desktop/BDA/NEON_Cleaned_By_Site/D02_POSE
✅ Saved: /home/vishnu/Desktop/BDA/NEON_Cleaned_By_Site/D02_PO

inputDir = /home/vishnu/Desktop/BDA/NEON_chem-surfacewater
outputDir = /home/vishnu/Desktop/BDA/NEON_Cleaned_By_Site
usefulFiles = List(swc_fieldData, swc_externalLabDataByAnalyte)
ignoredFiles = List(categoricalCodes, validation, variables, readme, EML, manifest, externalLabBlanks, externalLabSummaryData)


getAllFiles: (dir: java.io.File)Seq[java.io.File]
isUsefulCSV: (file: java.io.File)Boolean
extractSiteID: (fileName: String)Option[String]
findSiteColumn: (columns: Array[String])Option[String]
isUnwantedColumn: (column: String)Boolean
parseCSVLine: (line: String)Array[String]
escapeCSV: (value: String)String


List(categoricalCodes, validation, variables, readme, EML, manifest, externalLabBlanks, externalLabSummaryData)